<a href="https://colab.research.google.com/github/MinhNguyen19/honours_project/blob/main/code_submission/finetune_llama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import pandas as pd
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import torch
import time
from google.colab import files
from datasets import Dataset
import gdown
from unsloth import FastLanguageModel
import zipfile
import os

NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [ ]:
# Model with 1 epoch
file_id = '15PRnTjC8IpBZc0_GKFQtTnT9SOb3LSFo'
# Model with 2 epochs
file_id = '1XYMG_5X1Vs_YvrnBXAFDz8WwXm80mKNb'

epoch_url = f'https://drive.google.com/uc?id={file_id}'
gdown.download(epoch_url, "epoch.zip", quiet=False)

# Unzip file

with zipfile.ZipFile("epoch.zip", 'r') as zip_ref:
    zip_ref.extractall("epoch")

Downloading...
From (original): https://drive.google.com/uc?id=1XYMG_5X1Vs_YvrnBXAFDz8WwXm80mKNb
From (redirected): https://drive.google.com/uc?id=1XYMG_5X1Vs_YvrnBXAFDz8WwXm80mKNb&confirm=t&uuid=6d51cfcd-0c50-4513-96c7-6337c3ff5473
To: /content/epoch_2.zip
100%|██████████| 158M/158M [00:04<00:00, 38.4MB/s]


'epoch_2.zip'

In [ ]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name = "./epoch", # Use this to load epoch models
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2025.12.7: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [ ]:
# QLoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2025.12.7 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
# Import data

url_1 = 'https://raw.githubusercontent.com/MinhNguyen19/honours_project/refs/heads/main/datasets_ft/preprocessed_datasets/politics/LIAR.feather'
url_2 = 'https://raw.githubusercontent.com/MinhNguyen19/honours_project/refs/heads/main/datasets_ft/preprocessed_datasets/politics/pheme.feather'

response = requests.get(url_1)
df_liar = pd.read_feather(BytesIO(response.content))

response = requests.get(url_2)
df_pheme = pd.read_feather(BytesIO(response.content))

df_liar_copy = df_liar.copy()
df_pheme_copy = df_pheme.copy()

In [ ]:
politics_id = '1tX1dMUS4R3qAqrZBreqvEFQH1UgdCGF1'
politics_url = f'https://drive.google.com/uc?id={politics_id}'
gdown.download(politics_url, "politics.feather", quiet=False)

df_politics = pd.read_feather("politics.feather")
df_politics_title = df_politics.copy()
df_politics_title = df_politics_title.drop(columns=['text'])
df_politics_title['text'] = df_politics_title['metadata'].apply(lambda x: x['metadata']['title'])

Downloading...
From: https://drive.google.com/uc?id=1tX1dMUS4R3qAqrZBreqvEFQH1UgdCGF1
To: /content/politics.feather
100%|██████████| 31.7M/31.7M [00:00<00:00, 166MB/s]


In [ ]:
df_liar_filtered = df_liar[['text', 'label']]
df_pheme_filtered = df_pheme[['text', 'label']]
df_politics_title_filtered = df_politics_title[['text', 'label']]

# Merge datasets
df_combined = pd.concat([df_liar_filtered, df_pheme_filtered, df_politics_title_filtered], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
from sklearn.model_selection import train_test_split

# Get train, validation and test set

df_train_1, df_test = train_test_split(df_combined, test_size=0.2, random_state=42, stratify=df_combined['label'])

display(df_train_1['label'].value_counts())
display(df_test['label'].value_counts())

df_train, df_eval = train_test_split(df_train_1, test_size=0.1, random_state=42, stratify=df_train_1['label'])
display(df_train['label'].value_counts())
display(df_eval['label'].value_counts())

,count
label,
0,18429
1,14497


,count
label,
0,4608
1,3624


,count
label,
0,16586
1,13047


,count
label,
0,1843
1,1450


In [ ]:
from sklearn.utils import resample

# Separate majority and minority classes
df_train_majority = df_train[df_train.label == 0]
df_train_minority = df_train[df_train.label == 1]

# Undersample majority class
df_train_majority_undersampled = resample(
    df_train_majority,
    replace=False,
    n_samples=len(df_train_minority),
    random_state=42
)
df_train_balanced = pd.concat([df_train_majority_undersampled, df_train_minority])
display(df_train_balanced.label.value_counts())

,count
label,
0,13047
1,13047


In [ ]:
def format_for_sft(example):
    # Format for fine-tuning

    # label_str = "REAL" if example['label'] == 1 else "FAKE"
    label_str = example['label']
    system_prompt = "You are an expert news categorizer designed to classify claims as FAKE or REAL. Respond with one word only (FAKE/REAL)."

    user_prompt = (
        f"Determine whether the following news/claim is FAKE or REAL. \n"
        f"Answer with exactly one word: FAKE or REAL.\n"
        f"News: {example['text']}\n"
        f"Answer: "

    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": label_str}
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": formatted_text}

In [ ]:
ds_train = Dataset.from_pandas(df_train_balanced)
ds_combined_ft = ds_train.map(format_for_sft, remove_columns=['text', 'label'])
ds_eval = Dataset.from_pandas(df_eval)

Map:   0%|          | 0/26094 [00:00<?, ? examples/s]

In [ ]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import evaluate

# Load accuracy metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_preds):
    """
    Computes accuracy metrics from evaluation predictions.

    Args:
        eval_preds (tuple): A tuple containing logits and labels from the model.

    Returns:
        dict: A dictionary containing the accuracy score.
    """
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    mask = labels != -100
    labels = labels[mask]
    predictions = predictions[mask]

    return metric.compute(predictions=predictions, references=labels)

In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds_combined_ft,
    eval_dataset = ds_eval,
    compute_metrics = compute_metrics,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 16, # 64 for A100
        gradient_accumulation_steps = 8,
        eval_strategy = "epoch",
        warmup_steps = 5,
        num_train_epochs = 2,
        # max_steps = 144,
        save_strategy="steps",
        save_steps = 30,
        logging_steps = 10,
        learning_rate = 2e-4,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/26094 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/3293 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
torch.cuda.empty_cache()

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# Save full model + tokenizer
trainer.save_model("outputs/fine_tuned_model")
tokenizer.save_pretrained("outputs/fine_tuned_model")


('outputs/fine_tuned_model/tokenizer_config.json',
 'outputs/fine_tuned_model/special_tokens_map.json',
 'outputs/fine_tuned_model/chat_template.jinja',
 'outputs/fine_tuned_model/tokenizer.json')

In [ ]:
from google.colab import files
import shutil

# Zip the model folder
shutil.make_archive("fine_tuned_model", 'zip', "outputs/fine_tuned_model")

files.download("fine_tuned_model.zip")

'/content/fine_tuned_model.zip'

Testing


In [ ]:
df_true = df_train[df_train['label'] == 0]
df_false = df_train[df_train['label'] == 1]

In [ ]:
df_test['label'].value_counts()

,count
label,
0,4608
1,3624


In [ ]:
import random

def combine_examples(df_true, df_false, num_examples):
    random_num = random.randint(0,1)
    # random_num = 1

    true_examples = df_true.sample(num_examples + random_num)
    false_examples = df_false.sample(num_examples + 1 - random_num)

    combined_examples = pd.concat([true_examples, false_examples])
    return combined_examples

In [ ]:
def prompt_maker(text, examples=None):
    if examples is None:
        return_text = (
        f"Determine whether the following news/claim is REAL or FAKE. \n"
        f"Answer with exactly one word (FAKE or REAL).\n"
        # f"Classify the news as REAL or FAKE. Output only a number, 0 or 1. 0=REAL, 1=FAKE\n"
        f"News: {text}\n"
        f"Answer: "
        )

        return return_text
    else:
        fewshot = ""
        i=1
        for index, row in examples.iterrows():
            r_text = row['text']
            r_label = row['label']
            if r_label == 'FAKE':
                r_label = '0'
            else:
                r_label = '1'
            fewshot += f"News: \"{row['text']}\"\nAnswer: {row['label']}\n\n"
            i+=1
        fewshot = fewshot.strip()
        return (
            "Classify the news as REAL or FAKE. Output only a number, 0 or 1. 0=REAL, 1=FAKE\n"
            f"{fewshot}\n"
            "Now determine whether the following news/claim is FAKE or REAL. \n"
            f"News: \"{text}\"\nAnswer with exactly one number, 0 or 1\n"
            "Answer:"
        )

In [ ]:
def predict_batch(model, tokenizer, batch, fewshots, df_true, df_false, num_examples, examples, batch_size, max_new_tokens, max_length=512):
    predictions = []
    raw_outputs = []

    for i in range(0, len(batch["text"]), batch_size):
        sub_texts = batch["text"][i:i+batch_size]
        if fewshots:
            if examples is None:
                examples = combine_examples(df_true, df_false, num_examples)
            prompts = [
                prompt_maker(text, examples)
                for text in sub_texts
            ]
        else:
            prompts = [
                prompt_maker(text)
                for text in sub_texts
            ]
            # print(prompts)

        # Tokenize the small batch
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            truncation=True,
            padding="longest",
            max_length=max_length
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                  **inputs,
                  max_new_tokens=max_new_tokens,
                  temperature=0.0,
                  do_sample=False,
                  repetition_penalty=1.2,
                  eos_token_id=tokenizer.eos_token_id
              )
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        raw_outputs.extend(decoded)

        # Predictions for sub-batch
        for out, prompt in zip(decoded, prompts):
            cleaned = out[len(prompt):].strip()
            text_upper = cleaned.upper()
            # fake_count = text_upper.count("FAKE")
            # real_count = text_upper.count("REAL")
            fake_count = text_upper.count("1")
            real_count = text_upper.count("0")
            if fake_count > real_count:
                predictions.append(1)
            elif real_count > fake_count:
                predictions.append(0)
            else:
                predictions.append(-1)

    return {"prediction": predictions, "raw_output": raw_outputs}


In [ ]:
def predict_result(ds, fewshots=False, df_true=None, df_false=None, num_examples=1, examples=None, batch_size=30, max_new_tokens=20):
    """
    Predicts the labels for a given dataset using the loaded language model. A wrapper for predict_batch.

    Args:
        ds (Dataset): The dataset containing the text for prediction.
        fewshots (bool, optional): Whether to use few-shot prompting. Defaults to False.
        df_true (pd.DataFrame, optional): DataFrame of true examples for few-shot prompting. Required if fewshots is True.
        df_false (pd.DataFrame, optional): DataFrame of false examples for few-shot prompting. Required if fewshots is True.
        num_examples (int, optional): Number of examples to use for few-shot prompting. Defaults to 1. Number of few-shot = num_examples x2 + 1. E.g. numexamples = 1 => 3 shots
        examples (pd.DataFrame, optional): Pre-combined examples for few-shot prompting. If provided, df_true, df_false, and num_examples are ignored.
        batch_size (int, optional): The number of samples to process in each batch. Defaults to 30.
        max_new_tokens (int, optional): The maximum number of new tokens to generate for each prediction. Defaults to 20.

    Returns:
        dict: A dictionary containing two lists:
              - 'prediction': List of predicted labels (0 for REAL, 1 for FAKE, -1 for undecided).
              - 'raw_output': List of raw model outputs.
    """
    results = {"prediction": [], "raw_output": []}

    start_time = time.time()  # start timer

    for i in range(0, len(ds), batch_size):
        if i % 900 == 0:
            print(f"Reached {i+1} samples.")
        batch = ds[i:i+batch_size]  # process batch_size prompts at a time
        batch_result = predict_batch(model, tokenizer, batch, fewshots, df_true, df_false, num_examples, examples, batch_size, max_new_tokens)
        results["prediction"].extend(batch_result["prediction"])
        results["raw_output"].extend(batch_result["raw_output"])

    end_time = time.time()  # end timer
    elapsed_time = end_time - start_time

    print(f"Total inference time: {elapsed_time:.2f} seconds")
    print(f"Average time per row: {elapsed_time / len(ds):.2f} seconds")

    return results

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# Dataset containing examples that BERT and Mistral failed to identify correctly

fail_id = '19Xk-ZJprB-OdZBoNu_jANj-Bzt-5JjPR'
fail_url = f'https://drive.google.com/uc?id={fail_id}'
gdown.download(fail_url, "fail.csv", quiet=False)

df_fail = pd.read_csv("fail.csv")
df_fail_test = df_fail.copy()
df_fail_test['actual_label'] = df_fail_test['actual_label'].replace({'Fake': 1, 'Real': 0})

merged_df = pd.merge(df_fail_test, df_combined, on='text', how='inner', suffixes=('_fail_test', '_combined'))

NameError: name 'gdown' is not defined

In [ ]:
ds_test = Dataset.from_pandas(df_fail_test)

In [ ]:
# ds_test = Dataset.from_pandas(df_test)
ds_test = Dataset.from_pandas(df_test)

In [ ]:
results_zero = predict_result(ds_test, max_new_tokens=8)#, fewshots=True, df_true=df_true, df_false=df_false, num_examples=2)

Reached 1 samples.
Reached 901 samples.
Reached 1801 samples.
Reached 2701 samples.
Reached 3601 samples.
Reached 4501 samples.
Reached 5401 samples.
Reached 6301 samples.
Reached 7201 samples.
Reached 8101 samples.
Total inference time: 881.43 seconds
Average time per row: 0.11 seconds


In [ ]:
results_3shot = predict_result(ds_test, max_new_tokens=8, fewshots=True, df_true=df_true, df_false=df_false, num_examples=1)

Reached 1 samples.
Total inference time: 31.50 seconds
Average time per row: 0.17 seconds


In [ ]:
results_5shot = predict_result(ds_test, max_new_tokens=8, fewshots=True, df_true=df_true, df_false=df_false, num_examples=2)

Reached 1 samples.
Total inference time: 35.25 seconds
Average time per row: 0.19 seconds


In [ ]:
df_results = pd.DataFrame(results_zero)
df_results['prediction'].value_counts()

,count
prediction,
0,5040
1,2896
-1,296


In [ ]:
df_test

,text,label
33892,WOW! WHAT HAPPENED When Somebody Asked Beyonce...,1
35355,Unemployment in Texas has risen over two perce...,0
31421,Four state Assembly Democrats scored a death b...,1
18779,"The controversial history of France's ""Charlie...",0
10568,Argentine TV ad mocks Trump to promote soccer ...,0
...,...,...
26962,Says Oregon canned blueberries will be cheaper...,0
1459,The fiscal cliff deal ultimately raised taxes.,0
5712,DARTMOUTH #BlackLivesMatter Terrorists TEAR DO...,1
7890,"Maine governor rejects latest budget deal, rea...",0


In [ ]:
import pandas as pd

data_for_df = []

for i in range(len(ds_test)):
    true_label = ds_test['actual_label'][i]
    pred_zero = results_zero['prediction'][i]
    pred_3 = results_3shot['prediction'][i]
    pred_5 = results_5shot['prediction'][i]

    # Check condition: predictions are in agreement but differ from the true label
    if (pred_zero == pred_3 == pred_5) and (pred_zero != true_label) and (pred_zero != -1):
        data_for_df.append({
            'row_id': df_fail['row_id'][i],
            'actual_label': true_label,
            'llama_0shot': pred_zero,
            'llama_3shot': pred_3,
            'llama_5shot': pred_5
        })

df_analysis = pd.DataFrame(data_for_df)
df_analysis.head(20)

,row_id,actual_label,llama_0shot,llama_3shot,llama_5shot
0,36,1,0,0,0
1,45,1,0,0,0
2,50,1,0,0,0
3,94,1,0,0,0
4,100,1,0,0,0
5,102,1,0,0,0
6,169,1,0,0,0
7,173,1,0,0,0
8,189,1,0,0,0
9,213,1,0,0,0


In [ ]:
label_mapping = {0: 'Real', 1: 'Fake'}

df_analysis['actual_label'] = df_analysis['actual_label'].map(label_mapping)
df_analysis['llama_0shot'] = df_analysis['llama_0shot'].map(label_mapping)
df_analysis['llama_3shot'] = df_analysis['llama_3shot'].map(label_mapping)
df_analysis['llama_5shot'] = df_analysis['llama_5shot'].map(label_mapping)

print("DataFrame with labels replaced:")
display(df_analysis.head())

NameError: name 'df_analysis' is not defined

In [ ]:
df_combined_analysis = pd.merge(df_analysis, df_fail, on='row_id', how='inner')
df_combined_analysis = df_combined_analysis.drop(columns=['actual_label_y'])
df_combined_analysis.size

583

In [ ]:
if df_combined_analysis['actual_label_x'].equals(df_combined_analysis['actual_label_y']):

The columns are equal.


In [ ]:
df_combined_analysis.to_csv('df_combined_analysis.csv', index=False)
from google.colab import files
files.download('df_combined_analysis.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
new_list = []
for row in df_results.iterrows():
    if row[1]['prediction'] == -1:
        output = row[1]['raw_output']
        last_char = output[-1]
        if last_char == '0':
            new_list.append(0)
        elif last_char == '1':
            new_list.append(1)
        else:
            new_list.append(-1)
    else:
        new_list.append(row[1]['prediction'])

df_results['new_pred'] = new_list
df_results['new_pred'].value_counts()


,count
new_pred,
0,4797
1,3424
-1,11


In [ ]:
undecided_predictions = df_results[df_results['prediction'] == -1].head(1)
for row in undecided_predictions.iterrows():
    print(str(row[1]['prediction']) + row[1]['raw_output'])

-1Classify the news as REAL or FAKE. Output only a number, 0 or 1. 0=FAKE, 1=REAL.
News: "Strange flag in #Sydney, not the one affiliated with #ISIS. http://t.co/36m6A3AKSb"
Answer: 1

News: "According to #CharlieHebdo’s lawyer four well-known French cartoonists were killed by the masked gunmen: Cabu, Wolinski, Charb et Tignous."
Answer: 1

News: "Stay safe Ottawa."
Answer: 0

News: "Under President Barack Obama, more Americans are in poverty... than at any time since the Census Bureau began keeping records on it over 50 years ago."
Answer: 0

News: ""In the last 50 years, (the federal government has) only balanced the budget five times.""
Answer: 0
Now determine whether the following news/claim is FAKE or REAL. 
News: "When youre white... you dont know what its like to be poor."
Answer with exactly one number, 0 or 1
Answer: 0


In [ ]:
import zipfile
import os

# Unzip the uploaded file
with zipfile.ZipFile("fine_tuned_model.zip", 'r') as zip_ref:
    zip_ref.extractall("fine_tuned_model")

print("Model unzipped successfully to 'fine_tuned_model' directory.")

Model unzipped successfully to 'fine_tuned_model' directory.


In [ ]:
ds_test = Dataset.from_pandas(df_test)
print("df_test successfully converted to Hugging Face Dataset.")

df_test successfully converted to Hugging Face Dataset.


In [ ]:
from sklearn.metrics import classification_report

# Get the true labels from the original dataset
y_true = df_test["label"]

# Get the predicted labels from the results dictionary
y_pred = df_results["prediction"]


y_true.reset_index(drop=True, inplace=True)

# Filter out the undecided predictions (-1) and corresponding true labels
mask = [p != -1 for p in y_pred]
y_true_filtered = [y_true[i] for i in range(len(y_true)) if mask[i]]
y_pred_filtered = [y_pred[i] for i in range(len(y_pred)) if mask[i]]


# Generate and print the classification report
print(classification_report(y_true_filtered, y_pred_filtered))

              precision    recall  f1-score   support

           0       0.57      0.65      0.60      4397
           1       0.47      0.38      0.42      3539

    accuracy                           0.53      7936
   macro avg       0.52      0.51      0.51      7936
weighted avg       0.52      0.53      0.52      7936



In [ ]:
# ft zeroshot fake/real
#             precision    recall  f1-score   support

#            0       0.57      0.65      0.60      4397
#            1       0.47      0.38      0.42      3539

#     accuracy                           0.53      7936
#    macro avg       0.52      0.51      0.51      7936
# weighted avg       0.52      0.53      0.52      7936

In [ ]:
# finetune
# f"Determine whether the following news/claim is FAKE or REAL. \n"
#         f"Answer with exactly one word: FAKE or REAL.\n"
#         f"News: {example['text']}\n" # Use the 'text' column from the input data
#         f"Answer: "
# prompt
#  f"Classify the news as REAL or FAKE. Output only a number, 0 or 1. 0=FAKE, 1=REAL\n"
#         f"News: {text}\n"
#         f"Answer: "
# 0.25 epoch
# precision    recall  f1-score   support

#         FAKE       0.78      0.68      0.73      4461
#         REAL       0.66      0.77      0.71      3566

#     accuracy                           0.72      8027
#    macro avg       0.72      0.72      0.72      8027
# weighted avg       0.73      0.72      0.72      8027

In [ ]:
#  precision    recall  f1-score   support

#         FAKE       0.64      0.93      0.76      4484
#         REAL       0.80      0.33      0.47      3577

#     accuracy                           0.67      8061
#    macro avg       0.72      0.63      0.61      8061
# weighted avg       0.71      0.67      0.63      8061

#  precision    recall  f1-score   support

#         FAKE       0.61      0.93      0.74      4489
#         REAL       0.74      0.24      0.37      3575

#     accuracy                           0.63      8064
#    macro avg       0.67      0.59      0.55      8064
# weighted avg       0.67      0.63      0.57      8064

In [ ]:
# 0.5 epoch
#   precision    recall  f1-score   support

#         FAKE       0.57      0.99      0.72      4486
#         REAL       0.74      0.05      0.10      3577

#     accuracy                           0.57      8063
#    macro avg       0.65      0.52      0.41      8063
# weighted avg       0.64      0.57      0.44      8063

In [ ]:
# 1 epoch
# precision    recall  f1-score   support

#         FAKE       0.64      0.98      0.77      4490
#         REAL       0.93      0.30      0.45      3576

#     accuracy                           0.68      8066
#    macro avg       0.78      0.64      0.61      8066
# weighted avg       0.77      0.68      0.63      8066

In [ ]:
#  2 epoch
#  precision    recall  f1-score   support

#         FAKE       0.81      0.82      0.82      4480
#         REAL       0.77      0.76      0.76      3575

#     accuracy                           0.79      8055
#    macro avg       0.79      0.79      0.79      8055
# weighted avg       0.79      0.79      0.79      8055

In [ ]:
# NOTE: mixed up between 0 and 1
# Real = 0, Fake = 1 => This is correct